# 05. XGBoost 모델

## 목표
- XGBoost gradient boosting으로 일별 매출 예측
- Optuna 하이퍼파라미터 튜닝 (50 trials)
- SHAP Feature Importance 분석

## 접근 방식
- **Single Global Model**: 5개 상품군 데이터를 합치고 `family`를 label encoding하여 단일 모델 학습
- 기존 `feature_engineering.py`의 Lag/Rolling/EWM 피쳐 활용
- TimeSeriesSplit(5)으로 시계열 교차검증

In [ ]:
import sys
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import optuna
from sklearn.preprocessing import LabelEncoder

# 한글 폰트 설정
fm._load_fontmanager(try_read_cache=False)
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 프로젝트 모듈
sys.path.insert(0, "..")
from src.feature_engineering import create_all_features
from src.evaluation import evaluate_model
from src.models.xgboost_model import XGBoostModel

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Setup complete")

## 1. 데이터 로드 + 피쳐 엔지니어링

In [ ]:
# 전처리된 데이터 로드
train = pd.read_csv("../data/processed/train.csv", parse_dates=["date"])
val = pd.read_csv("../data/processed/val.csv", parse_dates=["date"])
test = pd.read_csv("../data/processed/test.csv", parse_dates=["date"])
top_families = pd.read_csv("../data/processed/top_families.csv")["family"].tolist()

# 전체 데이터 합쳐서 피쳐 생성 (lag 계산에 train 기간 데이터 필요)
full = pd.concat([train, val, test], ignore_index=True)
full = full.sort_values(["family", "date"]).reset_index(drop=True)

# Lag, Rolling, EWM 피쳐 생성
full = create_all_features(full)

print(f"Full data shape: {full.shape}")
print(f"Top 5 families: {top_families}")
print(
    f"\nNew features: {[c for c in full.columns if c.startswith(('lag_', 'rolling_', 'ewm_'))]}"
)
print(
    f"NaN count after feature engineering:\n{full.isnull().sum()[full.isnull().sum() > 0]}"
)

In [ ]:
# Family label encoding
le = LabelEncoder()
full["family_encoded"] = le.fit_transform(full["family"])
print(f"Family encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# 피쳐 컴럼 정의 (date, family, sales, day_name, holiday_type 제외)
exclude_cols = ["date", "family", "sales", "day_name", "holiday_type", "year"]
feature_cols = [c for c in full.columns if c not in exclude_cols]
print(f"\nFeature columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols):
    print(f"  {i + 1}. {col}")

In [ ]:
# 시간 기준으로 다시 분할
train_feat = full[full["date"] <= "2016-12-31"].copy()
val_feat = full[(full["date"] >= "2017-01-01") & (full["date"] <= "2017-06-30")].copy()
test_feat = full[full["date"] >= "2017-07-01"].copy()

# NaN 제거 (lag_365로 인한 첫 1년)
train_feat = train_feat.dropna(subset=feature_cols)
val_feat = val_feat.dropna(subset=feature_cols)
test_feat = test_feat.dropna(subset=feature_cols)

# X, y 분리
X_train = train_feat[feature_cols]
y_train = train_feat["sales"]
X_val = val_feat[feature_cols]
y_val = val_feat["sales"]
X_test = test_feat[feature_cols]
y_test = test_feat["sales"]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"\nTrain date range: {train_feat['date'].min()} ~ {train_feat['date'].max()}")
print(f"Val date range:   {val_feat['date'].min()} ~ {val_feat['date'].max()}")

## 2. 베이스라인 XGBoost (기본 파라미터)

먼저 기본 하이퍼파라미터로 XGBoost를 학습하여 베이스라인 성능을 확인한다.

In [ ]:
# 베이스라인 XGBoost (기본 파라미터)
baseline_model = XGBoostModel()
baseline_model.fit(X_train, y_train, X_val, y_val)
baseline_preds = baseline_model.predict(X_val)

# 상품군별 MAPE 계산
baseline_results = []
for family in top_families:
    mask = val_feat["family"] == family
    if mask.sum() == 0:
        continue
    result = evaluate_model(
        y_val[mask].values,
        baseline_preds[mask],
        model_name="XGBoost_Baseline",
        family=family,
        train_time=baseline_model.train_time_,
        predict_time=baseline_model.predict_time_,
    )
    baseline_results.append(result)

baseline_df = pd.DataFrame(baseline_results)
print("=== XGBoost Baseline (기본 파라미터) ===")
print(baseline_df[["model", "family", "mape", "rmse", "mae"]].to_string(index=False))
print(f"\nAvg MAPE: {baseline_df['mape'].mean():.2f}%")
print(f"Train time: {baseline_model.train_time_:.2f}s")

## 3. Optuna 하이퍼파라미터 튜닝

50회 trial로 최적 파라미터를 탐색한다. TimeSeriesSplit(5)으로 시계열 무결성을 유지하면서 교차검증한다.

### 탐색 공간
| 파라미터 | 범위 |
|----------|------|
| n_estimators | [100, 1000] |
| max_depth | [3, 10] |
| learning_rate | [0.01, 0.3] (log) |
| subsample | [0.6, 1.0] |
| colsample_bytree | [0.6, 1.0] |
| min_child_weight | [1, 10] |
| reg_alpha | [0, 10] |
| reg_lambda | [0, 10] |

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from src.evaluation import mape


def objective(trial):
    """Optuna objective: TimeSeriesSplit CV MAPE 최소화."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
        'random_state': 42,
        'n_jobs': -1,
    }

    # TimeSeriesSplit on training data
    tscv = TimeSeriesSplit(n_splits=5)
    mape_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_vl = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_vl = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
        preds = model.predict(X_vl)

        score = mape(y_vl.values, preds)
        if not np.isnan(score):
            mape_scores.append(score)

    return np.mean(mape_scores) if mape_scores else float('inf')


# Optuna 최적화 실행
study = optuna.create_study(direction='minimize', study_name='xgboost_tuning')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n=== Optuna 결과 ===")
print(f"Best MAPE (CV): {study.best_value:.2f}%")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# 최적화 과정 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial history
trials_df = study.trials_dataframe()
axes[0].plot(trials_df['number'], trials_df['value'], 'o-', alpha=0.6, markersize=4)
axes[0].axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.2f}%')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('MAPE (%)')
axes[0].set_title('Optuna 최적화 과정')
axes[0].legend()

# Parameter importance (top 5)
importances = optuna.importance.get_param_importances(study)
top_params = dict(list(importances.items())[:8])
axes[1].barh(list(top_params.keys())[::-1], list(top_params.values())[::-1])
axes[1].set_xlabel('Importance')
axes[1].set_title('하이퍼파라미터 중요도')

plt.tight_layout()
plt.savefig('../outputs/figures/18_optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 최적 모델 재학습 + Validation 평가

In [ ]:
# 최적 파라미터로 전체 train 데이터에 대해 재학습
best_params = {**study.best_params, 'random_state': 42, 'n_jobs': -1}
tuned_model = XGBoostModel(params=best_params)
tuned_model.fit(X_train, y_train, X_val, y_val)
tuned_preds = tuned_model.predict(X_val)

# 상품군별 평가
tuned_results = []
for family in top_families:
    mask = val_feat['family'] == family
    if mask.sum() == 0:
        continue
    result = evaluate_model(
        y_val[mask].values,
        tuned_preds[mask],
        model_name='XGBoost_Tuned',
        family=family,
        train_time=tuned_model.train_time_,
        predict_time=tuned_model.predict_time_,
    )
    tuned_results.append(result)

tuned_df = pd.DataFrame(tuned_results)
print("=== XGBoost Tuned (Optuna 최적) ===")
print(tuned_df[['model', 'family', 'mape', 'rmse', 'mae']].to_string(index=False))
print(f"\nAvg MAPE: {tuned_df['mape'].mean():.2f}%")
print(f"Train time: {tuned_model.train_time_:.2f}s")

# Baseline vs Tuned 비교
print("\n=== Baseline vs Tuned ===")
comparison = baseline_df[['family', 'mape']].rename(columns={'mape': 'baseline_mape'})
comparison = comparison.merge(tuned_df[['family', 'mape']].rename(columns={'mape': 'tuned_mape'}), on='family')
comparison['improvement'] = comparison['baseline_mape'] - comparison['tuned_mape']
print(comparison.to_string(index=False))